**Inspect Final Pretrain Dataset**

Checking the final dataset (i.e. version of colton minus A* but with traces)

In [14]:
import os
import json
import random
import torch
from collections import defaultdict
from tqdm.auto import tqdm

# =========================
# Config
# =========================
DATASET_PATH = "/workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded"
SEED = 42
SHOW_FIRST_N_SHARDS = 5
SAMPLES_PER_BACK = 3
TRACE_PREVIEW_LIMIT = None   # set to an int like 40 to truncate very long traces

rng = random.Random(SEED)

# =========================
# Helpers
# =========================
def shard_paths(path):
    if os.path.isdir(path):
        manifest = os.path.join(path, "manifest.json")
        if os.path.exists(manifest):
            with open(manifest) as f:
                names = json.load(f)["shards"]
            return [os.path.join(path, name) for name in names]
        return sorted(
            os.path.join(path, fn)
            for fn in os.listdir(path)
            if fn.startswith("shard_") and fn.endswith(".pt")
        )
    return [path]

def iter_trace_dataset(path):
    """
    Yields:
        {
            "dataset_idx": int,
            "prompt": str,
            "trace": str,
            "shard_path": str,
            "shard_local_idx": int,
        }
    """
    global_idx = 0
    for p in shard_paths(path):
        data = torch.load(p, weights_only=False, map_location="cpu")

        if not isinstance(data, dict):
            raise TypeError(f"Expected dict shard object in {p}, got {type(data)}")

        prompts = data["prompts"]
        traces = data["traces"]

        if len(prompts) != len(traces):
            raise ValueError(f"Length mismatch in {p}: {len(prompts)} prompts vs {len(traces)} traces")

        for local_idx, (prompt, trace) in enumerate(zip(prompts, traces)):
            yield {
                "dataset_idx": global_idx,
                "prompt": prompt,
                "trace": trace,
                "shard_path": p,
                "shard_local_idx": local_idx,
            }
            global_idx += 1

def split_trace_lines(trace):
    """
    Convert the single-line saved trace into readable lines.

    Example:
      "[ ... ] -> ... ; BACK -> ... ; SUCCESS ; EXPR (...)" 
    becomes a list of lines, preserving ';' where appropriate.
    """
    parts = [part.strip() for part in trace.split(";") if part.strip()]
    lines = []
    for part in parts:
        if part == "SUCCESS":
            lines.append("SUCCESS ;")
        elif part.startswith("EXPR"):
            lines.append(part)
        else:
            lines.append(part + " ;")
    return lines

def count_back(trace):
    return trace.count("BACK")

def count_steps(trace):
    return len(split_trace_lines(trace))

def pct(x, n):
    return 100.0 * x / n if n else 0.0

def reservoir_update(bucket_samples, seen_so_far, item, k, rng):
    """
    Standard reservoir sampling for one bucket.
    seen_so_far is the count AFTER including the current item.
    """
    if len(bucket_samples) < k:
        bucket_samples.append(item)
    else:
        j = rng.randint(1, seen_so_far)
        if j <= k:
            bucket_samples[j - 1] = item

def print_header(title, width=100, char="="):
    print(char * width)
    print(title.center(width))
    print(char * width)

def print_subheader(title, width=100, char="-"):
    print("\n" + char * width)
    print(title.center(width))
    print(char * width)

def format_int(x):
    return f"{x:,}"

# =========================
# Intro
# =========================
paths = shard_paths(DATASET_PATH)

print_header("Inspect pretraining trace dataset")
print(f"Dataset path        : {DATASET_PATH}")
print(f"Num shard files     : {len(paths):,}")
print(f"Seed                : {SEED}")
print(f"Samples / BACK bin  : {SAMPLES_PER_BACK}")
print(f"Sampling method     : random reservoir sampling within each BACK-count bucket")
print(f"Trace preview limit : {'FULL' if TRACE_PREVIEW_LIMIT is None else TRACE_PREVIEW_LIMIT}")
print()
print("First few shards:")
for p in paths[:SHOW_FIRST_N_SHARDS]:
    print("  ", p)

# Optional: show manifest metadata if present
manifest_path = os.path.join(DATASET_PATH, "manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print()
    print("Manifest metadata:")
    print(f"  format_version : {manifest.get('format_version')}")
    print(f"  trace_mode     : {manifest.get('trace_mode')}")
    vocab = manifest.get("vocab")
    if isinstance(vocab, list):
        print(f"  vocab size     : {len(vocab):,}")

# =========================
# Scan dataset
# =========================
checked = 0
total_prompt_chars = 0
total_trace_chars = 0
total_steps = 0

back_counts = defaultdict(int)
steps_sum_by_back = defaultdict(int)
trace_chars_sum_by_back = defaultdict(int)

# Per-bucket reservoir samples
samples_by_back = defaultdict(list)
seen_by_back = defaultdict(int)

with tqdm(desc="Inspecting trace dataset", unit="sample") as pbar:
    for r in iter_trace_dataset(DATASET_PATH):
        prompt = r["prompt"]
        trace = r["trace"]

        n_back = count_back(trace)
        n_steps = count_steps(trace)

        checked += 1
        total_prompt_chars += len(prompt)
        total_trace_chars += len(trace)
        total_steps += n_steps

        back_counts[n_back] += 1
        steps_sum_by_back[n_back] += n_steps
        trace_chars_sum_by_back[n_back] += len(trace)

        seen_by_back[n_back] += 1
        reservoir_update(
            samples_by_back[n_back],
            seen_by_back[n_back],
            {
                "dataset_idx": r["dataset_idx"],
                "prompt": prompt,
                "trace": trace,
                "n_back": n_back,
                "n_steps": n_steps,
                "trace_chars": len(trace),
                "shard_path": r["shard_path"],
                "shard_local_idx": r["shard_local_idx"],
            },
            SAMPLES_PER_BACK,
            rng,
        )

        pbar.update(1)

        if checked % 100000 == 0:
            pbar.set_postfix_str(
                f"checked={checked:,} current_back={n_back} steps={n_steps}",
                refresh=False,
            )

# =========================
# Summary
# =========================
print_subheader("Summary")

print(f"Checked samples      : {format_int(checked)}")
print(f"Avg prompt chars     : {total_prompt_chars / checked:.2f}")
print(f"Avg trace chars      : {total_trace_chars / checked:.2f}")
print(f"Avg steps / trace    : {total_steps / checked:.2f}")

print("\n=== BACK distribution ===")
for b in sorted(back_counts):
    c = back_counts[b]
    avg_steps_b = steps_sum_by_back[b] / c
    avg_trace_chars_b = trace_chars_sum_by_back[b] / c
    print(
        f"BACK={b:>2}: {c:>10,}  ({pct(c, checked):6.2f}%)"
        f" | avg steps={avg_steps_b:6.2f}"
        f" | avg trace chars={avg_trace_chars_b:8.2f}"
    )

# =========================
# Pretty sample printing
# =========================
for b in sorted(samples_by_back):
    bucket_samples = samples_by_back[b]
    available = back_counts[b]

    print("\n" + "=" * 100)
    print(f"BACK count = {b}".center(100))
    print(f"Available: {available:,} | Sampling: {len(bucket_samples)}".center(100))
    print("=" * 100)

    # Sort sampled examples by dataset index just for stable pretty display
    bucket_samples = sorted(bucket_samples, key=lambda x: x["dataset_idx"])

    for sample_idx, s in enumerate(bucket_samples, 1):
        print("\n" + "=" * 100)
        print(
            f"Category BACK={b} | Sample {sample_idx}/{len(bucket_samples)} "
            f"| dataset idx={s['dataset_idx']} | steps={s['n_steps']}"
        )
        print("- PROMPT:")
        print(s["prompt"])
        print("- TRACE:")

        lines = split_trace_lines(s["trace"])
        if TRACE_PREVIEW_LIMIT is not None:
            lines = lines[:TRACE_PREVIEW_LIMIT]

        for line in lines:
            print(line)

        if TRACE_PREVIEW_LIMIT is not None:
            original_lines = split_trace_lines(s["trace"])
            if len(original_lines) > TRACE_PREVIEW_LIMIT:
                print(f"... ({len(original_lines) - TRACE_PREVIEW_LIMIT} more lines omitted)")

        print("- META:")
        print(f"  trace chars     : {s['trace_chars']}")
        print(f"  shard local idx : {s['shard_local_idx']}")
        print(f"  shard file      : {os.path.basename(s['shard_path'])}")

                                 Inspect pretraining trace dataset                                  
Dataset path        : /workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded
Num shard files     : 70
Seed                : 42
Samples / BACK bin  : 3
Sampling method     : random reservoir sampling within each BACK-count bucket
Trace preview limit : FULL

First few shards:
   /workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded/shard_000000.pt
   /workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded/shard_000001.pt
   /workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded/shard_000002.pt
   /workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded/shard_000003.pt
   /workspace/sam/minireason_archives/data_pts/pretrain/randomized_target_traces_sharded/shard_000004.pt

Manifest metadata:
  format_version : 2
  trace_mode     : randomiz

Inspecting trace dataset: 0sample [00:00, ?sample/s]

Inspecting trace dataset: 6948231sample [01:45, 65737.67sample/s, checked=6,900,000 current_back=5 steps=22] 


----------------------------------------------------------------------------------------------------
                                              Summary                                               
----------------------------------------------------------------------------------------------------
Checked samples      : 6,948,231
Avg prompt chars     : 35.19
Avg trace chars      : 682.45
Avg steps / trace    : 15.28

=== BACK distribution ===
BACK= 0:    302,314  (  4.35%) | avg steps=  6.19 | avg trace chars=  272.98
BACK= 1:    605,348  (  8.71%) | avg steps=  8.51 | avg trace chars=  361.75
BACK= 2:  1,710,896  ( 24.62%) | avg steps= 11.33 | avg trace chars=  517.00
BACK= 3:    981,893  ( 14.13%) | avg steps= 13.91 | avg trace chars=  607.87
BACK= 4:  1,310,103  ( 18.86%) | avg steps= 16.67 | avg trace chars=  764.53
BACK= 5:    837,376  ( 12.05%) | avg steps= 19.26 | avg trace chars=  859.38
BACK= 6:    600,260  (  8.64%) | avg steps= 22.06 | avg trace chars=  981.94
BACK= 7: 